In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 16.8 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import SimpleITK as sitk
import numpy as np

Loading Data

In [5]:
def load_patient_images(ct_path, seg_path):
    ct_img = sitk.ReadImage(str(ct_path))
    seg_img = sitk.ReadImage(str(seg_path))

    return ct_img, seg_img

Geometry Alignment

In [6]:
def align_seg_to_ct(ct_img, seg_img):

    seg_resampled = sitk.Resample(
        seg_img,
        ct_img,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        seg_img.GetPixelID()
    )

    return seg_resampled

Convert to Numpy

In [7]:
def seg_to_numpy(seg_img):

    seg_np = sitk.GetArrayFromImage(seg_img)

    return seg_np

Vertebrae Extraction

In [8]:
def extract_vertebra_mask(seg_np, vertebra_label):

    mask = seg_np == vertebra_label

    return mask

Bounding Box Extraction

In [9]:
def compute_bbox(mask):
    if not np.any(mask):
        raise ValueError("Empty vertebra mask.")

    coords = np.where(mask)

    zmin, zmax = coords[0].min(), coords[0].max()
    ymin, ymax = coords[1].min(), coords[1].max()
    xmin, xmax = coords[2].min(), coords[2].max()

    return (
        zmin, zmax,
        ymin, ymax,
        xmin, xmax
    )

Margin Expansion: to preserve near anatomical features

In [10]:
def add_margin(
    bbox,
    volume_shape,
    margin_x=10,
    margin_y=10,
    margin_z=5
):
    zmin, zmax, ymin, ymax, xmin, xmax = bbox

    # Clamp minimum boundaries to 0
    xmin = max(0, xmin - margin_x)
    ymin = max(0, ymin - margin_y)
    zmin = max(0, zmin - margin_z)

    # Clamp maximum boundaries to (shape - 1) to prevent out-of-bounds indexing
    xmax = min(volume_shape[2] - 1, xmax + margin_x)
    ymax = min(volume_shape[1] - 1, ymax + margin_y)
    zmax = min(volume_shape[0] - 1, zmax + margin_z)

    # Return as standard integers for SimpleITK compatibility
    return (int(zmin), int(zmax), int(ymin), int(ymax), int(xmin), int(xmax))

CT Cropping: remove irrelevant anatomy

In [11]:
def crop_ct_image(ct_img, bbox):

    zmin, zmax, ymin, ymax, xmin, xmax = bbox

    index = [
        int(xmin),
        int(ymin),
        int(zmin)
    ]

    size = [
        int(xmax - xmin + 1),
        int(ymax - ymin + 1),
        int(zmax - zmin + 1)
    ]

    cropped_img = sitk.RegionOfInterest(
        ct_img,
        size=size,
        index=index
    )

    return cropped_img

Isotropic Resampling: standardize voxel size


In [12]:
def resample_to_isotropic(ct_crop_img, target_spacing=(1.0, 1.0, 1.0)):
    """
    Resample a cropped CT image to isotropic voxel spacing.

    Parameters
    ----------
    ct_crop_img : SimpleITK.Image
        Cropped vertebral CT image.

    target_spacing : tuple
        Desired voxel spacing (default = 1x1x1 mm).

    Returns
    -------
    ct_iso_img : SimpleITK.Image
        Isotropically resampled CT image.
    """

    old_spacing = ct_crop_img.GetSpacing()
    old_size = ct_crop_img.GetSize()

    new_size = [
        int(round(old_size[i] * old_spacing[i] / target_spacing[i]))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()

    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)

    resampler.SetOutputDirection(
        ct_crop_img.GetDirection()
    )

    resampler.SetOutputOrigin(
        ct_crop_img.GetOrigin()
    )

    resampler.SetInterpolator(
        sitk.sitkLinear
    )

    ct_iso_img = resampler.Execute(ct_crop_img)

    return ct_iso_img

CT to Numpy

In [13]:
def ct_to_numpy(ct_img):

    ct_np = sitk.GetArrayFromImage(ct_img)

    return ct_np

HU Windowing: suppress soft tissue

In [14]:
def apply_bone_window(
    volume,
    hu_min=-450,
    hu_max=1050
):
    return np.clip(
        volume,
        hu_min,
        hu_max
    )

Normalize Intensity: for stabilized input for Neural Network

In [15]:
def normalize_intensity(
    volume,
    hu_min=-450,
    hu_max=1050
):

    volume = (
        volume - hu_min
    ) / (
        hu_max - hu_min
    )

    volume = (
        volume * 2
    ) - 1

    return volume

Resize to Network Input

In [16]:
import numpy as np
import SimpleITK as sitk


def resize_to_network_input(
    volume,
    reference_spacing=(1.0, 1.0, 1.0),
    target_size=(64, 64, 64)
):
    """
    Resize a normalized isotropic CT volume to the
    fixed network input size.

    Parameters
    ----------
    volume : numpy.ndarray
        Normalized isotropic CT volume.

    reference_spacing : tuple
        Current voxel spacing.
        After isotropic resampling this is (1,1,1).

    target_size : tuple
        Desired network input size.

    Returns
    -------
    numpy.ndarray
        Volume of shape (64,64,64)
    """

    img = sitk.GetImageFromArray(
        volume.astype(np.float32)
    )

    img.SetSpacing(reference_spacing)

    resampler = sitk.ResampleImageFilter()

    resampler.SetSize(target_size)

    new_spacing = [
        img.GetSize()[i]
        * img.GetSpacing()[i]
        / target_size[i]
        for i in range(3)
    ]

    resampler.SetOutputSpacing(new_spacing)

    resampler.SetInterpolator(
        sitk.sitkLinear
    )

    img_resized = resampler.Execute(img)

    volume_resized = sitk.GetArrayFromImage(
        img_resized
    )

    return volume_resized

## Final Complete Wrapper Function

In [17]:
import numpy as np

def preprocess_single_vertebra(
    ct_img,
    seg_np,
    vertebra_label,
    target_spacing=(1.0, 1.0, 1.0),
    target_size=(64, 64, 64),
):
    """
    Preprocess one vertebra using pre-loaded patient volumes.
    """
    # Extract vertebra
    vertebra_mask = extract_vertebra_mask(seg_np, vertebra_label)

    # Bounding box
    bbox = compute_bbox(vertebra_mask)

    # Add margins (using full CT size to prevent out-of-bounds)
    bbox = add_margin(bbox, seg_np.shape)

    # Crop CT
    ct_crop_img = crop_ct_image(ct_img, bbox)

    # Resample to isotropic
    ct_iso_img = resample_to_isotropic(ct_crop_img, target_spacing)

    # CT → NumPy
    ct_np = ct_to_numpy(ct_iso_img).astype(np.float32)

    # Windowing
    ct_np = apply_bone_window(ct_np)

    # Normalize
    ct_np = normalize_intensity(ct_np)

    # Resize
    ct_np = resize_to_network_input(
        ct_np,
        reference_spacing=ct_iso_img.GetSpacing(),
        target_size=target_size
    )

    return ct_np.astype(np.float32)

## Test with 1 Patient

In [24]:
#test
PATIENT_ID = "15067"
DATASET_ROOT = Path("/content/drive/MyDrive/TCIA_Original/nifti_images/Spine-Mets-CT-SEG-NIFTI")

patient_dir = DATASET_ROOT / PATIENT_ID

ct_path = patient_dir / "15067_ct.nii.gz"
seg_path = patient_dir / "15067_seg-1.nii.gz"

In [25]:
tensor = preprocess_single_vertebra(
    ct_path=ct_path,
    seg_path=seg_path,
    vertebra_label=5
)

print(tensor.shape)
print(tensor.dtype)
print(tensor.min())
print(tensor.max())

(224, 224, 224)
float32
-1.0
1.0


In [26]:
import matplotlib.pyplot as plt

rows = 4
cols = 4
slices_per_page = rows * cols

for start in range(0, tensor.shape[0], slices_per_page):

    fig, axes = plt.subplots(rows, cols, figsize=(10, 10))

    for j, ax in enumerate(axes.flat):

        idx = start + j

        if idx < tensor.shape[0]:
            ax.imshow(tensor[idx], cmap="gray")
            ax.set_title(f"{idx}", fontsize=9)
            ax.axis("off")
        else:
            ax.axis("off")

    fig.suptitle(f"15067 - T5 | Slices {start}–{min(start+slices_per_page-1, tensor.shape[0]-1)}")
    plt.tight_layout()
    plt.show()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

===========================================================================================================

Create dataset and sequence

In [24]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

Setting up directory

In [23]:
DATA_DIR = Path("/content/drive/MyDrive/TCIA_Original/nifti_images/Spine-Mets-CT-SEG-NIFTI")
OUT_DIR = Path("/content/drive/MyDrive/TCIA_Original/tensors")
OUT_DIR.mkdir(parents=True, exist_ok=True)



Vertebrae Label Mapping: Always consistent

In [26]:
VERT_MAP = {
    'T1': 1, 'T2': 2, 'T3': 3, 'T4': 4, 'T5': 5, 'T6': 6,
    'T7': 7, 'T8': 8, 'T9': 9, 'T10': 10, 'T11': 11, 'T12': 12,
    'L1': 13, 'L2': 14, 'L3': 15, 'L4': 16, 'L5': 17
}
INV_VERT_MAP = {v: k for k, v in VERT_MAP.items()}

Load the normalized CSV

In [27]:
df = pd.read_csv("/content/drive/MyDrive/TCIA_Data/vertebra_labels_claude.csv")

In [32]:
df.head()

,Case,Vertebra,Class
0,10250,T2,Mixed
1,10352,ALL,No Lesion
2,10355,T1,Mixed
3,10355,T2,Mixed
4,10355,T3,Mixed


In [28]:
patient_col = 'Case'
vert_col = 'Vertebra'
class_col = 'Class'


In [34]:
master_labels = []

In [60]:
grouped_patients = df.groupby(patient_col)

# ==========================================
# 3. MASTER PATIENT LOOP
# ==========================================
for patient_id, group in tqdm(grouped_patients, desc="Processing Patients"):

    # Construct paths to this specific patient's CT and SEG NIfTI files
    ct_path = DATA_DIR / str(patient_id) / f"{patient_id}_ct.nii.gz"
    seg_path = DATA_DIR / str(patient_id) / f"{patient_id}_seg-1.nii.gz"

    if not ct_path.exists() or not seg_path.exists():
        print(f"Skipping {patient_id}: Missing CT or SEG file.")
        continue

    # --- 3A. Load and Align (Done ONCE per patient) ---
    try:
        # Load raw images from disk into RAM
        ct_img, seg_img = load_patient_images(ct_path, seg_path)
        # Fix geometry mismatch so SEG voxels perfectly overlap CT voxels
        seg_resampled = align_seg_to_ct(ct_img, seg_img)
        # Convert SEG to a NumPy array for fast mathematical querying
        seg_np = seg_to_numpy(seg_resampled)
    except Exception as e:
        print(f"Error loading {patient_id}: {e}")
        continue

    # Detect exactly which vertebrae masks actually exist in this patient's scan
    present_labels = np.unique(seg_np)

    # ==========================================
    # 4. VERTEBRAE EXTRACTION LOOP
    # ==========================================
    # Loop through the CSV rows assigned to this specific patient
    for _, row in group.iterrows():
        target_vert = row['Vertebra']
        lesion_class = row['Class']

        verts_to_process = []

        # --- 4A. Determine which bones to process ---
        if target_vert == 'ALL':
            # Patient is Normal: Grab every single vertebra present in the mask
            verts_to_process = [lbl for lbl in present_labels if lbl in INV_VERT_MAP]
        else:
            # Patient is Diseased: Only grab the specific bone listed in this row
            target_vert = target_vert.strip()
            if target_vert in VERT_MAP:
                lbl = VERT_MAP[target_vert]
                if lbl in present_labels:
                    verts_to_process = [lbl]

        # --- 4B. Process and Save Output ---
        for vert_label in verts_to_process:
            vert_name = INV_VERT_MAP[vert_label]
            out_filename = f"{patient_id}_{vert_name}.npy"
            out_filepath = OUT_DIR / out_filename

            # Resume Check: Skip if file already exists from a previous crashed run
            if out_filepath.exists():
                master_labels.append({'filename': out_filename, 'label': lesion_class})
                continue

            try:
                # Execute the fast, refactored preprocessing function
                tensor_64 = preprocess_single_vertebra(
                    ct_img=ct_img,
                    seg_np=seg_np,
                    vertebra_label=vert_label
                )

                # Save the 64x64x64 tensor to disk
                np.save(out_filepath, tensor_64)
                # Log the file and its class for the PyTorch DataLoader
                master_labels.append({'filename': out_filename, 'label': lesion_class})

            except Exception as e:
                print(f"Error processing {patient_id} {vert_name}: {e}")


# ==========================================
# 5. GENERATE FINAL DATASET INDEX
# ==========================================
# Convert the logged data into a DataFrame and save as CSV
labels_df = pd.DataFrame(master_labels)
labels_df.to_csv("dataset/labels.csv", index=False)

Processing Patients: 100%|██████████| 55/55 [17:05<00:00, 18.64s/it]


### Reprocessing Specific Patients

The following section processes only the patient IDs that were previously skipped and appends their generated labels to the `labels.csv` file.

In [31]:
missing_ids = ['14826', '13653', '14797']


In [33]:
DATA_DIR = Path("/content/drive/MyDrive/TCIA_Original/nifti_images/Spine-Mets-CT-SEG-NIFTI")
OUT_DIR = Path("/content") # Saving directly to Colab root

# Load your starting CSV
df = pd.read_csv("/content/drive/MyDrive/TCIA_Data/vertebra_labels_claude.csv")

# Filter to only the 3 missing patients
missing_group = df[df['Case'].astype(str).isin(missing_ids)].groupby('Case')

for patient_id, group in missing_group:
    patient_id = str(patient_id)
    ct_path = DATA_DIR / patient_id / f"{patient_id}_ct.nii.gz"
    seg_path = DATA_DIR / patient_id / f"{patient_id}_seg-1.nii.gz"

    print(f"\n========== Loading Patient {patient_id} ==========")

    try:
        ct_img, seg_img = load_patient_images(ct_path, seg_path)
        seg_np = seg_to_numpy(align_seg_to_ct(ct_img, seg_img))
        present_labels = np.unique(seg_np)
        print(f"Bones actually found in NIfTI mask: {present_labels}")
    except Exception as e:
        print(f"CRITICAL ERROR: Failed to load NIfTI files: {e}")
        continue

    for _, row in group.iterrows():
        raw_vert = str(row['Vertebra'])
        lesion_class = row['Class']

        # Aggressive cleanup
        target_vert = raw_vert.replace('.', '').replace(';', '').strip()
        print(f"\n  Checking row: CSV said '{raw_vert}' -> Cleaned to '{target_vert}'")

        if target_vert in VERT_MAP:
            lbl = VERT_MAP[target_vert]
            if lbl in present_labels:
                out_filename = f"{patient_id}_{target_vert}.npy"
                out_filepath = OUT_DIR / out_filename

                try:
                    tensor_64 = preprocess_single_vertebra(ct_img, seg_np, lbl)
                    np.save(out_filepath, tensor_64)
                    print(f"  ✅ SUCCESS: Saved {out_filename} to {out_filepath}")
                except Exception as e:
                    print(f"  ❌ ERROR during cropping/resampling: {e}")
            else:
                print(f"  ⚠️ WARNING: Bone '{target_vert}' (Label {lbl}) is NOT in the patient's segmentation mask!")
        else:
            print(f"  ⚠️ WARNING: '{target_vert}' is not a recognized bone name.")


========== Loading Patient 13653 ==========
Bones actually found in NIfTI mask: [0]

  Checking row: CSV said 'T12' -> Cleaned to 'T12'
  ⚠️ WARNING: Bone 'T12' (Label 12) is NOT in the patient's segmentation mask!

  Checking row: CSV said 'L1' -> Cleaned to 'L1'
  ⚠️ WARNING: Bone 'L1' (Label 13) is NOT in the patient's segmentation mask!

  Checking row: CSV said 'L5' -> Cleaned to 'L5'
  ⚠️ WARNING: Bone 'L5' (Label 17) is NOT in the patient's segmentation mask!

========== Loading Patient 14797 ==========
Bones actually found in NIfTI mask: [ 0 14 15 16 17]

  Checking row: CSV said 'T12' -> Cleaned to 'T12'
  ⚠️ WARNING: Bone 'T12' (Label 12) is NOT in the patient's segmentation mask!

========== Loading Patient 14826 ==========
Bones actually found in NIfTI mask: [ 0 14 15 16 17]

  Checking row: CSV said 'T8' -> Cleaned to 'T8'
  ⚠️ WARNING: Bone 'T8' (Label 8) is NOT in the patient's segmentation mask!

  Checking row: CSV said 'L1' -> Cleaned to 'L1'
  ⚠️ WARNING: Bone 'L1' 